In [1]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

import numpy as np

In [15]:
data_raw = (
    pd.read_excel('data/sample_for_test (2).xlsx')
    .dropna()
)

data = pd.DataFrame()
data_raw.sample(11)

,text,sentiment
975,После присоединения новых регионов к России жи...,позитив
175,Леонид Попробуйте зайти на страницу основной и...,нейтрал
468,Величие было у СССР А у РФ его нет и не будет ...,негатив
1769,Здоровья Вам великая русская женщина,позитив
319,Любовь ну если тот володя Тот тот не как муха ...,нейтрал
1503,Юрий Ну тогда и Пименова с Макаренковым,нейтрал
1188,Вот промах в те времена такого маникюра не было,нейтрал
1891,Храни Вас Господь дорогие воины,позитив
2138,Спасибо гн Сладков Помним Ирочку В Мариуполе р...,позитив
2760,Обманутый убийцабудет наказанна чужой земле,негатив


In [16]:
rename_dict = {
    'нейтрал': 0,    
    'позитив': 1,    
    'негатив': -1,  
    'негативно': -1,
    'нейтрально': 0,      
    'позит': 1,      
    'нейтр': 0,        
    'нег': 0,       
    'пози': 1,
    'позитивный': 1
}

data['sentiment'] = data_raw['sentiment'].replace(rename_dict)

In [ ]:
data_raw['text'] = data_raw['text'].str.strip()


data['repeated_chars_count'] = data_raw['text'].str.count(r'(.)\1{3,}') # повторяющие буквы аааа
data['has_digit_prefix'] = data_raw['text'].str.match(r'^\d{3,}').astype(int) # числа в начале

data['text_clean'] = (
    data_raw['text']
    # 1. убрать цифровой мусор в начале строки
    .str.replace(r'^\d+\s*', '', regex=True)
    # 2. убрать *%?: мусор (цензура матов)
    .str.replace(r'[\*%\?:]{2,}', ' ', regex=True)
    # 3. нормализовать повторы букв: ааааа → аа
    .str.replace(r'(.)\1{3,}', r'\1\1', regex=True)
    # 4. схлопнуть множественные пробелы в один
    .str.replace(r' {2,}', ' ', regex=True)
    # 5. убрать пробелы по краям
    .str.strip()
)


data['text_len'] = data['text_clean'].str.len()

# Количество слов
data['word_count'] = data['text_clean'].str.split().str.len()

# Средняя длина слова: суммируем длины слов / кол-во слов
data['avg_word_len'] = (
    data['text_clean']
    .str.split()
    .apply(lambda words: pd.Series(words).str.len().mean() if words else 0)
)

# Доля уникальных слов: уникальные / все
data['unique_word_ratio'] = (
    data['text_clean']
    .str.lower()
    .str.split()
    .apply(lambda words: len(set(words)) / len(words) if words else 0)
)

In [21]:
data

,sentiment,repeated_chars_count,has_digit_prefix,text_clean,text_len,word_count,avg_word_len,unique_word_ratio
0,1,0,0,Толя конечно Герой Успехов во всём и здоровья ...,110,16,5.937500,1.000000
1,0,1,1,у нас искусственный интеллект сейчас даже там ...,183,31,4.935484,0.903226
2,1,0,0,Мне радостно что наше многогранное Отечество п...,99,13,6.692308,1.000000
3,-1,0,0,Ну и говно,10,3,2.666667,1.000000
4,1,0,0,@ Спасибо подписался,20,3,6.000000,1.000000
...,...,...,...,...,...,...,...,...
3031,0,0,0,Ты смотрела сериалДесять дней до весны,38,6,5.500000,1.000000
3032,1,0,0,Пусть Ангелы оберегают наших воинов,35,5,6.200000,1.000000
3033,-1,0,0,Клоун не может быть политиком с илу скудтумия,45,8,4.750000,1.000000
3034,1,0,0,Ирина согласна Но досмотрю до конца И всем сов...,50,9,4.666667,1.000000


### llm сегмент

In [1]:
with open('../giga_api_key.txt', 'r', encoding='utf-8') as f:
        auth_key = f.read().strip()

In [65]:
def parse_text_sentiment(text):

    import time
    from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
    from langchain_gigachat.chat_models import GigaChat
    from langchain_classic.prompts import ChatPromptTemplate
    time.sleep(1)

    with open('../giga_api_key.txt', 'r', encoding='utf-8') as f:
        auth_key = f.read().strip()

    llm = GigaChat(
        credentials=auth_key,
        model="GigaChat",
        temperature=0.4,
        verify_ssl_certs=False
    )

    positive_word_count_schema = ResponseSchema(
        name="positive_word_count",
        description='''
        Верни ТОЛЬКО целое число — количество слов с позитивной окраской: похвала, радость, одобрение.
        Пример: 3
        Если таких слов нет — верни 0.
        '''
    )

    negative_word_count_schema = ResponseSchema(
        name="negative_word_count",
        description='''
        Верни ТОЛЬКО целое число — количество слов с негативной окраской: ругань, обида, агрессия.
        Пример: 2
        Если таких слов нет — верни 0.
        '''
    )

    neutral_word_count_schema = ResponseSchema(
        name="neutral_word_count",
        description='''
        Верни ТОЛЬКО целое число — количество нейтральных слов без эмоциональной окраски.
        Пример: 5
        Если таких слов нет — верни 0.
        '''
    )

    negation_count_schema = ResponseSchema(
        name="negation_count",
        description='''
        Верни ТОЛЬКО целое число — количество отрицаний: не, нет, нельзя, никогда и похожих.
        Пример: 1
        Если таких слов нет — верни 0.
        '''
    )

    intensifier_count_schema = ResponseSchema(
        name="intensifier_count",
        description='''
        Верни ТОЛЬКО целое число — количество усилителей эмоции: очень, крайне, абсолютно, дико, огонь и похожих.
        Пример: 2
        Если таких слов нет — верни 0.
        '''
    )


    emotional_shift_schema = ResponseSchema(
        name="emotional_shift",
        description='''
        1 если тональность текста резко меняется внутри одного сообщения
        (например, начинается позитивно, заканчивается негативно или наоборот), иначе 0.
        Пример: 1
        '''
    )

    aggression_flag_schema = ResponseSchema(
        name="aggression_flag",
        description='''
        1 если текст содержит оскорбления, грубую лексику или агрессивные конструкции, иначе 0.
        Пример: 1
        '''
    )

    emotional_intensity_schema = ResponseSchema(
        name="emotional_intensity",
        description='''
        Верни целое число от 0 до 3 — общий градус эмоциональности текста:
        0 — полностью нейтральный,
        1 — слабо эмоциональный,
        2 — эмоциональный,
        3 — очень эмоциональный (крик, ярость, восторг).
        Пример: 2
        '''
    )

    rhetorical_question_count_schema = ResponseSchema(
        name="rhetorical_question_count",
        description='''
        Верни ТОЛЬКО целое число — количество риторических вопросов.
        Риторический вопрос НЕ требует ответа и используется для усиления эффекта.
        Примеры риторических: «Разве это не абсурд?», «Кому вообще это нужно?»
        Примеры НЕ риторических: «Где посмотреть?», «Что думаешь?»
        Если таких конструкций нет — верни 0.
        Пример: 1

        Если таких конструкций нет — верни 0.
        '''
    )
    
    appeal_flag_schema = ResponseSchema(
        name="appeal_flag",
        description='''
        1 если текст содержит прямое обращение к аудитории или конкретному человеку
        (ребята, братья, вы, ты, друзья и похожие), иначе 0.
        Пример: 1
        '''
    )

    aggression_flag_schema = ResponseSchema(
        name="aggression_flag",
        description='''
        1 если текст содержит угрозы или явные призывы к агрессивным действиям, иначе 0.
        Пример: 0
        '''
    )

    religious_lexicon_flag_schema = ResponseSchema(
        name="religious_lexicon_flag",
        description='''
        1 если текст содержит религиозную лексику, включая молитвенные обращения и императивные конструкции: 
        Бог, Аллах, Храни, молитва, благослови и похожие, иначе 0.
        Пример: 1
        '''
    )

    political_lexicon_flag_schema = ResponseSchema(
        name="political_lexicon_flag",
        description='''
        1 если текст содержит политическую лексику: названия стран, организаций (НАТО, Украина, Россия),
        политические термины (санкции, война, президент и похожие), иначе 0.
        Пример: 1
        '''
    )

    target_entity_schema = ResponseSchema(
        name="target_entity",
        description='''
        На кого направлен основной эмоциональный посыл текста.
        Верни одно из значений: "person", "group", "country", "abstract", "none".
        person — конкретный человек,
        group — группа людей (ребята, солдаты, журналисты),
        country — государство или организация,
        abstract — абстрактное явление (война, правда, демократия),
        none — посыл не выражен.
        Пример: "group"
        '''
    )

    response_schemas = [
        positive_word_count_schema,
        negative_word_count_schema,
        neutral_word_count_schema,
        negation_count_schema,
        intensifier_count_schema,
        emotional_shift_schema,
        aggression_flag_schema,
        emotional_intensity_schema,
        rhetorical_question_count_schema,
        appeal_flag_schema,
        religious_lexicon_flag_schema,
        political_lexicon_flag_schema,
        target_entity_schema,

    ]

    parser = StructuredOutputParser.from_response_schemas(response_schemas)
    format_instructions = parser.get_format_instructions()

    review_template = """
    Ты лингвист. Проанализируй русскоязычный текст и посчитай маркеры тональности.
    Верни только JSON, без пояснений. Все числовые поля — только цифры, без слов.

    {format_instructions}

    Текст: {text}
    """

    prompt_template = ChatPromptTemplate.from_template(review_template)
    messages = prompt_template.format_messages(
        text=text,
        format_instructions=format_instructions
    )

    response = llm.invoke(messages)

    try:
        output_dict = parser.parse(response.content)
    except Exception:
        output_dict = {
            "positive_word_count": 0,
            "negative_word_count": 0,
            "neutral_word_count": 0,
            "negation_count": 0,
            "intensifier_count": 0,
            "emotional_shift": None,
            "aggression_flag": None,
            "emotional_intensity": None,
            "rhetorical_question_count": 0,
            "appeal_flag": None,
            "religious_lexicon_flag": None,
            "political_lexicon_flag": None,
            "target_entity": None,
        }

    return output_dict

In [67]:
data['json_parse'] = data['text_clean'].progress_apply(lambda text: parse_text_sentiment(text))

100%|██████████| 3030/3030 [2:07:04<00:00,  2.52s/it]  


In [69]:
data.to_csv('data/clear_parse_json.csv', index = None)